# Experiment 4.4.1 — Stage-2 Memory Architecture Sweep

Analysis-only notebook for the finalized `tau_mem × recurrence` sweep.

Primary architecture metric: **Stage-2 endpoint membrane (`Uend`) + train-only scaled Linear probe BA**.

Secondary metrics:
- Hidden WholeCount + Linear BA
- Output WholeCount BA
- `Uend - HiddenCount` and `Uend - OutputCount` gaps
- firing rate and post-end tail activity
- paired `RSNN - FF` effect at each Stage-2 time constant


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root")

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / "notebooks" / "artifacts" / "experiment_4_4_1_stage2_memory_sweep" / "stage2_tau_x_recurrence_v1"
FILES = {
    "runs": ROOT / "runs.csv",
    "summary": ROOT / "summary.csv",
    "paired": ROOT / "paired_effects.csv",
    "paired_summary": ROOT / "paired_effects_summary.csv",
    "manifest": ROOT / "manifest.json",
}
missing = [str(path) for path in FILES.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing finalized Exp4.4.1 artifacts:\n" + "\n".join(missing))

runs = pd.read_csv(FILES["runs"])
summary = pd.read_csv(FILES["summary"])
paired = pd.read_csv(FILES["paired"])
paired_summary = pd.read_csv(FILES["paired_summary"])
manifest = json.loads(FILES["manifest"].read_text())
display(manifest)


## Protocol completeness


In [ ]:
expected_taus = {125, 250, 500, 1000, 2000}
expected_models = {"ff", "rsnn"}
expected_seeds = {11, 23, 37, 53, 71}
assert set(manifest["tau_mem_ms"]) == expected_taus
assert set(manifest["model_kinds"]) == expected_models
assert set(manifest["seeds"]) == expected_seeds
assert manifest["run_count"] == 50
test = runs[runs["split"] == "test"].copy()
coverage = test.groupby(["model_kind", "tau_mem_ms"])["seed"].agg(lambda s: set(s))
for model in expected_models:
    for tau in expected_taus:
        assert coverage.loc[(model, tau)] == expected_seeds
print("Protocol coverage OK: 50 independent runs, 5 paired seeds per condition.")
display(test.head())


## Aggregate test metrics


In [ ]:
cols = [
    "model_kind", "tau_mem_ms",
    "uend_linear_ba_mean", "uend_linear_ba_std",
    "hidden_whole_count_linear_ba_mean", "hidden_whole_count_linear_ba_std",
    "output_whole_count_ba_mean", "output_whole_count_ba_std",
    "uend_minus_hidden_count_ba_mean",
    "uend_minus_output_count_ba_mean",
]
display(summary[cols].sort_values(["model_kind", "tau_mem_ms"]))


## Primary architecture metric — Uend BA vs Stage-2 tau


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for model in ["ff", "rsnn"]:
    sub = summary[summary["model_kind"] == model].sort_values("tau_mem_ms")
    ax.errorbar(sub["tau_mem_ms"], sub["uend_linear_ba_mean"], yerr=sub["uend_linear_ba_std"], marker="o", capsize=4, label=model.upper())
ax.set_xscale("log", base=2)
ax.set_xticks([125, 250, 500, 1000, 2000])
ax.set_xticklabels(["125", "250", "500", "1000", "2000"])
ax.set_xlabel("Stage-2 tau_mem (ms)")
ax.set_ylabel("Test balanced accuracy")
ax.set_title("Exp4.4.1: gesture-level endpoint memory")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Readout matrix across tau


In [ ]:
readouts = [("uend_linear_ba_mean", "Uend + Linear"), ("hidden_whole_count_linear_ba_mean", "HiddenCount + Linear"), ("output_whole_count_ba_mean", "Output WholeCount")]
for model in ["ff", "rsnn"]:
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    sub = summary[summary["model_kind"] == model].sort_values("tau_mem_ms")
    for col, label in readouts:
        ax.plot(sub["tau_mem_ms"], sub[col], marker="o", label=label)
    ax.set_xscale("log", base=2)
    ax.set_xticks([125, 250, 500, 1000, 2000])
    ax.set_xticklabels(["125", "250", "500", "1000", "2000"])
    ax.set_xlabel("Stage-2 tau_mem (ms)")
    ax.set_ylabel("Test balanced accuracy")
    ax.set_title(f"{model.upper()}: memory vs spike-accessible readouts")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()


## State-to-spike / output information gaps


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for model in ["ff", "rsnn"]:
    sub = summary[summary["model_kind"] == model].sort_values("tau_mem_ms")
    ax.plot(sub["tau_mem_ms"], sub["uend_minus_output_count_ba_mean"], marker="o", label=f"{model.upper()}: Uend - OutputCount")
ax.axhline(0.0, linewidth=1)
ax.set_xscale("log", base=2)
ax.set_xticks([125, 250, 500, 1000, 2000])
ax.set_xticklabels(["125", "250", "500", "1000", "2000"])
ax.set_xlabel("Stage-2 tau_mem (ms)")
ax.set_ylabel("BA gap")
ax.set_title("Information exposure gap")
ax.legend()
ax.grid(alpha=0.25)
plt.show()
display(summary[["model_kind", "tau_mem_ms", "uend_minus_hidden_count_ba_mean", "uend_minus_hidden_count_ba_std", "uend_minus_output_count_ba_mean", "uend_minus_output_count_ba_std"]].sort_values(["model_kind", "tau_mem_ms"]))


## Paired recurrence contribution


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for readout, label in [("uend_linear", "Uend + Linear"), ("hidden_whole_count_linear", "HiddenCount + Linear"), ("output_whole_count", "Output WholeCount")]:
    sub = paired_summary[paired_summary["readout"] == readout].sort_values("tau_mem_ms")
    ax.errorbar(sub["tau_mem_ms"], sub["mean"], yerr=sub["std"], marker="o", capsize=4, label=label)
ax.axhline(0.0, linewidth=1)
ax.set_xscale("log", base=2)
ax.set_xticks([125, 250, 500, 1000, 2000])
ax.set_xticklabels(["125", "250", "500", "1000", "2000"])
ax.set_xlabel("Stage-2 tau_mem (ms)")
ax.set_ylabel("Paired BA delta (RSNN - FF)")
ax.set_title("Recurrence contribution as a function of passive memory")
ax.legend()
ax.grid(alpha=0.25)
plt.show()
display(paired_summary.sort_values(["readout", "tau_mem_ms"]))


## Activity / stability diagnostics


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for model in ["ff", "rsnn"]:
    sub = summary[summary["model_kind"] == model].sort_values("tau_mem_ms")
    ax.plot(sub["tau_mem_ms"], sub["state_tail_event_fraction_mean"], marker="o", label=f"{model.upper()} Stage-2 tail")
ax.set_xscale("log", base=2)
ax.set_xticks([125, 250, 500, 1000, 2000])
ax.set_xticklabels(["125", "250", "500", "1000", "2000"])
ax.set_xlabel("Stage-2 tau_mem (ms)")
ax.set_ylabel("Post-end tail event fraction")
ax.set_title("Stage-2 stability across memory time constants")
ax.legend()
ax.grid(alpha=0.25)
plt.show()
display(summary[["model_kind", "tau_mem_ms", "state_events_per_neuron_second_mean", "output_events_per_neuron_second_mean", "state_tail_event_fraction_mean", "output_tail_event_fraction_mean"]].sort_values(["model_kind", "tau_mem_ms"]))


## Ranking and interpretation table


In [ ]:
ranking = summary.copy()
ranking["memory_rank"] = ranking["uend_linear_ba_mean"].rank(method="min", ascending=False)
ranking["deployment_rank"] = ranking["output_whole_count_ba_mean"].rank(method="min", ascending=False)
ranking = ranking.sort_values(["memory_rank", "deployment_rank"])
display(ranking[["model_kind", "tau_mem_ms", "uend_linear_ba_mean", "uend_linear_ba_std", "memory_rank", "hidden_whole_count_linear_ba_mean", "output_whole_count_ba_mean", "output_whole_count_ba_std", "deployment_rank", "uend_minus_output_count_ba_mean", "state_tail_event_fraction_mean"]])
